In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import logging
import warnings
warnings.filterwarnings("ignore")
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
logging.disable(logging.CRITICAL)


In [ ]:
from deepcell.applications import Mesmer
from deepcell.utils.plot_utils import create_rgb_image, make_outline_overlay
import numpy as np
import cv2
import math 
import pandas as pd
import scanpy as sc
import skimage
import imagecodecs
import tifffile as tf
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from skimage import restoration
import distinctipy
from pathlib import Path 
import pyarrow
import dask
import dask.array as da
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
from skimage.exposure import rescale_intensity
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import matplotlib.pyplot as plt
import scipy
import igraph
import leidenalg

In [ ]:
os.environ.update({"DEEPCELL_ACCESS_TOKEN": ""})

In [ ]:
structure_marker_dict = {'Nuclei': ['DAPI']}

In [ ]:
def construct_marker_channel_dict(marker_list):
    marker_channel_dict = {}
    for marker in marker_list:
        marker_channel_dict[marker] = marker_list.index(marker)

    return marker_channel_dict

In [ ]:
all_marker_list = ['DAPI']
marker_list = ['DAPI']

In [ ]:
marker_channel_dict = construct_marker_channel_dict(marker_list)

In [ ]:
def normalize_img(chunk, out_range, in_range):

    normalized_img = rescale_intensity(chunk, in_range=in_range, out_range=out_range)
    return normalized_img

In [ ]:
def threshold_img(img, value, img_max):
    img_to_thr = img.copy()

    for channel in range(img_to_thr.shape[0]):
        img_channel = img_to_thr[channel, :, :]
        _, channel_mask = cv2.threshold(img_channel, 0, img_max, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        img_channel[channel_mask==0] = value
        img_to_thr[channel, :, :] = img_channel 

    return img_to_thr

In [ ]:
def extract_img_channels(img_layer, structure, img_range, cell=False):
    extracted_img_channels = []

    if(cell):
        channel_markers = cell_type_marker_dict[structure]
    else:
        channel_markers = structure_marker_dict[structure]
    for marker in channel_markers:
        img_channel_for_marker = img_layer.data[marker_channel_dict[marker], :, :]
        normalized_img_channel =normalize_img(img_channel_for_marker, img_range)
        extracted_img_channels.append(normalized_img_channel)
    
    combined_img_channels = extracted_img_channels[0]

    for channel_index in range(1, len(extracted_img_channels)):
        combined_img_channels = cv2.add(combined_img_channels, extracted_img_channels[channel_index])

    normalized_combined_img_channels = normalize_img(combined_img_channels, img_range)
    
    return normalized_combined_img_channels

In [ ]:
def format_img_for_segmentation(nuclei_channel_chunk, membrane_channel_chunk=None, nuclei=False):
    if(nuclei):
        empty_membrane_channel_chunk = da.from_array(np.zeros(nuclei_channel_chunk.shape, dtype=np.float32))
        combined_nuclei_membrane_chunk = da.stack([nuclei_channel_chunk, empty_membrane_channel_chunk], axis=-1)
    else:
        combined_nuclei_membrane_chunk = da.stack([nuclei_channel_chunk, membrane_channel_chunk], axis=-1)
    
    return combined_nuclei_membrane_chunk


In [ ]:
def cell_segmentation(combined_nuclei_membrane_chunk, segmentation_obj, nuclei=False, distance=1):

    expanded_combined_nuclei_membrane_chunk = da.expand_dims(combined_nuclei_membrane_chunk, axis=0)
    if(nuclei==True):
        segmented_chunk = skimage.segmentation.expand_labels(segmentation_obj.predict(expanded_combined_nuclei_membrane_chunk, image_mpp=0.5), distance)

    else:
        segmented_chunk = segmentation_obj.predict(expanded_combined_nuclei_membrane_chunk, image_mpp=0.5)
    

    return segmented_chunk
  


In [ ]:
import networkx as nx

def merge_cell_labels(labels, tile_h, tile_w, min_contact=5):
 
    uf = nx.utils.UnionFind()
    nrows = labels.shape[0] // tile_h
    ncols = labels.shape[1] // tile_w

    for row in range(nrows):
        for col in range(ncols):
            y0, x0 = row * tile_h, col * tile_w
            tile = labels[y0:y0+tile_h, x0:x0+tile_w]

            if col < ncols - 1:
                right_tile = labels[y0:y0+tile_h, (col+1)*tile_w:(col+2)*tile_w]
                a_edge, b_edge = tile[:, -1], right_tile[:, 0]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

            if row < nrows - 1:
                bottom_tile = labels[(row+1)*tile_h:(row+2)*tile_h, x0:x0+tile_w]
                a_edge, b_edge = tile[-1, :], bottom_tile[0, :]

                touching = (a_edge > 0) & (b_edge > 0)
                if np.any(touching):
                    a_vals, b_vals = a_edge[touching], b_edge[touching]
                    for a in np.unique(a_vals):
                        for b in np.unique(b_vals):
                            n_touch = np.sum((a_edge == a) & (b_edge == b))
                            if n_touch >= min_contact:
                                uf.union(int(a), int(b))

    mapping, next_id = {}, 1
    for lbl in np.unique(labels):
        if lbl == 0:
            continue
        root = uf[lbl]
        if root not in mapping:
            mapping[root] = next_id
            next_id += 1
    remap = {lbl: mapping[uf[lbl]] for lbl in np.unique(labels) if lbl != 0}

    merged = labels.copy()
    mask = merged > 0
    merged[mask] = np.vectorize(remap.get)(merged[mask])

    return merged

In [ ]:
def estimate_min_contact(labels, tile_h, tile_w, percentile=10):
    contact_counts = []

    nrows = labels.shape[0] // tile_h
    ncols = labels.shape[1] // tile_w

    for row in range(nrows):
        for col in range(ncols):
            y0, x0 = row*tile_h, col*tile_w
            tile = labels[y0:y0+tile_h, x0:x0+tile_w]

            if col < ncols - 1:
                right_tile = labels[y0:y0+tile_h, (col+1)*tile_w:(col+2)*tile_w]
                left_edge, right_edge = tile[:, -1], right_tile[:, 0]
                touching = (left_edge>0) & (right_edge>0)
                a_vals, b_vals = left_edge[touching], right_edge[touching]
                for a in np.unique(a_vals):
                    for b in np.unique(b_vals):
                        n_touch = np.sum((left_edge==a) & (right_edge==b))
                        contact_counts.append(n_touch)

            if row < nrows - 1:
                bottom_tile = labels[(row+1)*tile_h:(row+2)*tile_h, x0:x0+tile_w]
                top_edge, bottom_edge = tile[-1, :], bottom_tile[0, :]
                touching = (top_edge>0) & (bottom_edge>0)
                a_vals, b_vals = top_edge[touching], bottom_edge[touching]
                for a in np.unique(a_vals):
                    for b in np.unique(b_vals):
                        n_touch = np.sum((top_edge==a) & (bottom_edge==b))
                        contact_counts.append(n_touch)

    contact_counts = np.array(contact_counts)

    if len(contact_counts) == 0:
        return 1 

    min_contact = max(1, int(np.percentile(contact_counts, percentile)))

    return min_contact

## Run Cell Segmentation

In [ ]:
key = "" # sample identifier

In [ ]:
path_to_save = r"" # path to save registration metrics 

In [ ]:
images_path = r"" # path to registered images

In [ ]:
channel_paths = [f for f in Path(images_path + key).rglob('*.tif') if 'registered' in str(f) and 'Cycle' in str(f)]

channels = {}

for channel in channel_paths:
    print(channel)
    if('CH1' in str(channel)):
        cycle_index = str(channel).find('Cycle')
        cycle = str(channel)[cycle_index: cycle_index + len('Cycle')+2]
        
        channels[cycle] = da.from_array(tf.imread(channel), chunks=(512, 512))
    

In [ ]:
def resize_chunk(chunk, factor):
    return cv2.resize(chunk, None, fx=1/factor, fy=1/factor)

In [ ]:
channel_size = []
for channel_name, channel in channels.items(): 
    channel_size.append(channel.shape)

In [ ]:
img_min, img_max = dask.compute(da.min(channel), da.max(channel))

In [ ]:
channels_normalized = {}
for channel_name, channel in channels.items():
    channels_normalized[channel_name] = channel.map_blocks(normalize_img, (0, 255), (img_min, img_max))

In [ ]:
tile_combined_location_img_list = []
for i in range(0, list(channels_normalized.values())[0].shape[0], 512):
    for j in range(0, list(channels_normalized.values())[0].shape[1], 512):
        tile_combined_location_img_list.append((i, j))


In [ ]:
def compute_overlay_chunk(chunk_idx, chunk_list):
    return np.squeeze(chunk_list[chunk_idx].compute())

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter, label, center_of_mass


def make_tissue_mask(instance_mask, sigma=100):
    # Label IDs
    labels = np.arange(1, instance_mask.max() + 1)

    # Centroids (C implementation)
    centroids = np.asarray(
        center_of_mass(
            np.ones(instance_mask.shape, dtype=np.uint8),
            labels=instance_mask,
            index=labels,
        )
    )

    # Remove labels that are not present
    valid = ~np.isnan(centroids[:, 0])
    labels = labels[valid]

    ys = np.rint(centroids[valid, 0]).astype(np.int32)
    xs = np.rint(centroids[valid, 1]).astype(np.int32)

    # Density image
    density = np.zeros(instance_mask.shape, dtype=np.float32)
    density[ys, xs] = 1

    density = gaussian_filter(density, sigma=sigma)

    tissue_mask = density > skimage.filters.threshold_otsu(density)

    cc, n = label(tissue_mask)
    top_idx = np.argsort(np.bincount(cc.ravel())[1:])[-1:]
    top_labels = top_idx + 1
    tissue_mask = np.isin(cc, top_labels)

    contours, _ = cv2.findContours(tissue_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    filled_mask = np.zeros_like(filled_mask.astype(np.uint8))
    filled_mask = cv2.drawContours(filled_mask, contours, 0, 255, thickness=cv2.FILLED)

    # Labels whose centroids lie inside tissue
    keep = labels[filled_mask[ys, xs]]

    # Fast lookup table instead of looping over labels
    lut = np.zeros(instance_mask.max() + 1, dtype=bool)
    lut[keep] = True

    instance_mask_within_tissue = instance_mask.copy()
    instance_mask_within_tissue[~lut[instance_mask]] = 0

    return instance_mask_within_tissue, filled_mask

In [ ]:
segmentation_obj = Mesmer()

In [ ]:
for cycle_name, channel in channels_normalized.items():
    print(cycle_name)

In [ ]:
"""with open(path_to_save + '/' + key + '/segmentation_mask_no_processing_' + 'Cycle 1' + '.pickle', 'rb') as f:
    merged_segmentation_mask_matrix = pickle.load(f)"""

In [ ]:
for cycle_name, channel in channels_normalized.items():
    print(cycle_name)
    nuclei_img_for_segmentation = channel.map_blocks(format_img_for_segmentation, nuclei=True, chunks=(512, 512, 2), new_axis=[2])
    segmented_img = nuclei_img_for_segmentation.map_blocks(cell_segmentation, segmentation_obj, True, 1, dtype=np.int64, chunks=(1, 512, 512, 1), new_axis=[0])
    segmented_img_chunks = segmented_img.blocks.ravel()
    segmented_img_chunks_computed = [compute_overlay_chunk(block_idx, segmented_img_chunks) for block_idx in tqdm(range(len(segmented_img_chunks)))]

    segmentation_mask = []
    img_h = nuclei_img_for_segmentation.shape[0]
    img_w = nuclei_img_for_segmentation.shape[1]
    tile_h = 512
    tile_w = 512
    j = 0
    for i in range(int(img_h/tile_h)):
        segmentation_mask.append(np.hstack(segmented_img_chunks_computed[j:j+int(img_w/tile_w)]))
        j+= int(img_w/tile_w)
            
    segmentation_mask_matrix = np.vstack(segmentation_mask)

    segmentation_mask_offset = np.zeros(segmentation_mask_matrix.shape)

    offset = 0
    for l in range(len(tile_combined_location_img_list)):
        i, j = tile_combined_location_img_list[l]
        segmented_tile = segmentation_mask_matrix[i:i+512, j:j+512] 
        segmented_tile_non_background_mask = segmented_tile!=0 
        segmented_tile_offset = segmented_tile.copy()
        segmented_tile_offset[segmented_tile_non_background_mask] = segmented_tile[segmented_tile_non_background_mask] + offset
        segmentation_mask_offset[i:i+512, j:j+512] = segmented_tile_offset
        offset+=np.max(segmented_tile)
    merged_segmentation_mask_matrix = merge_cell_labels(segmentation_mask_offset, 512, 512, estimate_min_contact(segmentation_mask_offset, 512, 512))
    merged_segmentation_mask_matrix = merged_segmentation_mask_matrix.astype(int)

    merged_segmentation_mask_matrix_within_tissue, _ = make_tissue_mask(merged_segmentation_mask_matrix)
    print(path_to_save + '/' + key + '/segmentation_mask_' + cycle_name + '.pickle')
    with open(path_to_save + '/' + key + '/segmentation_mask_no_processing_' + cycle_name + '.pickle', 'wb') as f:
            pickle.dump(merged_segmentation_mask_matrix, f)
    with open(path_to_save + '/' + key + '/segmentation_mask_' + cycle_name + '.pickle', 'wb') as f:
        pickle.dump(merged_segmentation_mask_matrix_within_tissue, f)
    with open(path_to_save + '/' + key + '/segmentation_mask_binary_' + cycle_name + '.pickle', 'wb') as f:
            pickle.dump(_, f)
